In [ ]:
import pandas as pd

In [ ]:
data_train = pd.read_csv('train.csv')
data_test = pd.read_csv('test.csv')

In [ ]:
data_train

In [ ]:
data_train.iloc[0]


In [ ]:
data_train.isna().sum()

In [ ]:
data_train.duplicated().sum()

In [ ]:
data_train.head()

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder


# ??? ??????? ?????? ?? ?? ????
texts_train = (data_train['Description'].fillna('') + ' ' + data_train['Title'].fillna('')).tolist()
texts_test = (data_test['Description'].fillna('') + ' ' + data_test['Title'].fillna('')).tolist()



In [ ]:
max_words = 20000  # ???? ??? ????? ?? ???????
max_len = 100      # ??? ?????? ?????? (padding)

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts_train)

X_train_seq = tokenizer.texts_to_sequences(texts_train)
X_test_seq = tokenizer.texts_to_sequences(texts_test)

# ??? ?? ????? ???? ?????
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')


In [ ]:
# ?????? TargetColumn ???? ?????? ???? ????? ???????
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(data_train['Class Index'])

# ????? ??? One-hot
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train)

In [ ]:

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.5))
model.add(Dense(y_train_cat.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()



In [ ]:
from sklearn.model_selection import train_test_split

# Split the training data into training and validation sets
X_train_split, X_val_pad, y_train_split, y_val = train_test_split(
    X_train_pad, y_train_cat, test_size=0.2, random_state=42
)


history = model.fit(
    X_train_split, y_train_split,
    epochs=7,
    batch_size=64,
    validation_data=(X_val_pad, y_val)
)

In [ ]:
y_pred = model.predict(X_test_pad)


In [ ]:
y_pred_classes = np.argmax(y_pred, axis=1)

# ??? ???? test labels ??????
if 'Class Index' in data_test.columns:
    y_test = label_encoder.transform(data_test['Class Index'])

    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    print("Accuracy:", accuracy_score(y_test, y_pred_classes))
    print("Classification Report:\n", classification_report(y_test, y_pred_classes))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_classes))

In [ ]:
# ????? ??? ????? ?? ????? ??? notebook ?????? ???? ???????? ???????
import pickle
import joblib

# ??? ???????
model.save('models/lstm_classifier.h5')

# ??? ??? tokenizer
with open('models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# ??? ??? label encoder
with open('models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

# ??? ????????? ??????
config = {
    'max_words': max_words,
    'max_len': max_len,
    'class_names': {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}  # ??? ????????
}

with open('models/config.pkl', 'wb') as f:
    pickle.dump(config, f)

print("?? ??? ???? ???????? ?????!")